# C2 classifier — DeBERTa-v3-base fine-tune on Colab

Tiny launcher. The actual training code lives in `mostargate/classifier/train.py` in the repo; this notebook just bootstraps a Colab T4 instance and runs that module.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU` (free tier).
2. `Runtime → Run all`.
3. When done, download `classifier_model.zip` from the **Files** panel on the left and unzip into `dataset/classifier_artifacts/model/` locally.

Expected wall time on T4: 3–5 minutes including model download.

## 1. Pull the repo

In [31]:
!rm -rf mostargate
!git clone https://github.com/0xballistics/mostargate.git
%cd mostargate


Cloning into 'mostargate'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (158/158), done.
remote: Total 224 (delta 117), reused 159 (delta 60), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 2.71 MiB | 14.04 MiB/s, done.
Resolving deltas: 100% (117/117), done.
/content/mostargate/mostargate/mostargate/mostargate/mostargate/mostargate


## 2. Install Python deps

Colab images already include `torch` and `numpy`. We add the few packages `mostargate.classifier.train` imports beyond that.

In [32]:
!pip install -q transformers datasets sentencepiece protobuf python-dotenv

## 3. Verify GPU is attached

In [33]:
!nvidia-smi

Sun Jun 21 21:26:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             31W /   70W |    1543MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 4. Train

Hyperparameters are baked into `train.py` per `docs/phase_c_classifier_findings.md` §2 — 5 epochs, batch 16 (GPU), lr 2e-5, max_len 256, seed 42, BCEWithLogitsLoss with per-class pos_weight clipped at 10. No internal validation hold-out: all 500 records train the model, save the final checkpoint after epoch 5.

The first run downloads `microsoft/deberta-v3-base` (~700 MB) from the HuggingFace hub. Subsequent re-runs in the same session use the cached download.

In [39]:
!python -m mostargate.classifier.train --epochs 40 --learning-rate 2e-5

device=cuda batch=16 epochs=40 lr=2e-05 seed=42 fp16=False
Loaded 500 training records.
Label matrix: shape=(500, 15) dtype=float32 mean_positives_per_record=2.50
Loading weights: 100% 197/197 [00:00<00:00, 5083.00it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on yo

##5. Zip the trained model for download

Right-click `classifier_model.zip` in the file panel on the left → `Download`. Unzip locally so the directory ends up at `dataset/classifier_artifacts/model/`.

In [40]:
!zip -qr classifier_model.zip dataset/classifier_artifacts/model/
!ls -lh classifier_model.zip
print("\nDownload classifier_model.zip from the file panel (left sidebar).")

-rw-r--r-- 1 root root 407M Jun 21 21:50 classifier_model.zip

Download classifier_model.zip from the file panel (left sidebar).


## 6. Test model after training

In [41]:
import json, torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from mostargate.constants import TOOLS

PERMISSIONS = list(TOOLS.keys())
MODEL_DIR  = "dataset/classifier_artifacts/model"

tok   = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).cuda().eval()

test  = json.load(open("dataset/test.json"))
enc   = tok([r["prompt"] for r in test], padding=True, truncation=True,
            max_length=256, return_tensors="pt").to("cuda")
with torch.no_grad():
    probs = torch.sigmoid(model(**enc).logits).cpu().numpy()   # (100, 15)

# 1. Distribution of probabilities — bimodal means it learned, mush near 0.5 means it didn't
print(f"shape={probs.shape}  mean={probs.mean():.3f}  std={probs.std():.3f}\n")
for lo, hi in [(0,.05),(.05,.20),(.20,.40),(.40,.60),(.60,.80),(.80,.95),(.95,1.01)]:
    n   = ((probs >= lo) & (probs < hi)).sum()
    pct = 100*n/probs.size
    print(f"[{lo:.2f}, {hi:.2f}): {pct:5.1f}%  {'#'*int(pct/2)}")

# 2. Per-permission F1 at threshold 0.5
gt = np.array([[r["permissions"].get(p, False) for p in PERMISSIONS] for r in test], dtype=int)
preds = (probs >= 0.5).astype(int)
print(f"\n{'permission':<22}{'gt+':>5}{'pred+':>7}{'TP':>4}{'FP':>4}{'FN':>4}{'P':>6}{'R':>6}{'F1':>6}")
for j, p in enumerate(PERMISSIONS):
    tp = ((gt[:,j]==1) & (preds[:,j]==1)).sum()
    fp = ((gt[:,j]==0) & (preds[:,j]==1)).sum()
    fn = ((gt[:,j]==1) & (preds[:,j]==0)).sum()
    P  = tp / max(tp+fp, 1); R = tp / max(tp+fn, 1)
    F1 = 2*P*R / max(P+R, 1e-9)
    print(f"{p:<22}{gt[:,j].sum():>5}{preds[:,j].sum():>7}{tp:>4}{fp:>4}{fn:>4}{P:>6.2f}{R:>6.2f}{F1:>6.2f}")

macro_f1 = np.mean([
    (lambda tp,fp,fn: 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0)(
        ((gt[:,j]==1) & (preds[:,j]==1)).sum(),
        ((gt[:,j]==0) & (preds[:,j]==1)).sum(),
        ((gt[:,j]==1) & (preds[:,j]==0)).sum(),
    )
    for j in range(15)
])
print(f"\nmacro-F1 at threshold 0.5: {macro_f1:.3f}")
print(f"(reference: TF-IDF=0.70, Claude=0.89)")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

shape=(100, 15)  mean=0.208  std=0.322

[0.00, 0.05):  47.3%  #######################
[0.05, 0.20):  30.5%  ###############
[0.20, 0.40):   3.9%  #
[0.40, 0.60):   2.3%  #
[0.60, 0.80):   2.1%  #
[0.80, 0.95):   4.5%  ##
[0.95, 1.01):   9.3%  ####

permission              gt+  pred+  TP  FP  FN     P     R    F1
github_read              29     32  27   5   2  0.84  0.93  0.89
pull_request_create       1      1   1   0   0  1.00  1.00  1.00
code_execute              7      6   5   1   2  0.83  0.71  0.77
confluence_read          30     32  27   5   3  0.84  0.90  0.87
jira_read                19     14  13   1   6  0.93  0.68  0.79
jira_write               22     22  20   2   2  0.91  0.91  0.91
slack_read               10     11  10   1   0  0.91  1.00  0.95
slack_write               6      5   4   1   2  0.80  0.67  0.73
salesforce_read          12     12  11   1   1  0.92  0.92  0.92
database_read            44     54  43  11   1  0.80  0.98  0.88
email_read               20     23  

## 7. Risk based evaluation

In [42]:
import json, numpy as np
from pathlib import Path
from mostargate.constants import TOOLS, TOOL_TIERS

PERMISSIONS = list(TOOLS.keys())

# Cache the prediction matrix from what we just computed in the diagnostic
# (`probs` is already defined from the diagnostic cell — if not, re-run that first)
Path("dataset/classifier_artifacts").mkdir(parents=True, exist_ok=True)
np.save("dataset/classifier_artifacts/roberta_test_probs.npy", probs)
print(f"Saved probability matrix → roberta_test_probs.npy ({probs.shape})")

def tier_thr(p, t1, t2, t3):
    return {1: t1, 2: t2, 3: t3}[TOOL_TIERS[p]]

CONFIGS = {
    "static_05":             {p: 0.5 for p in PERMISSIONS},
    "static_08":             {p: 0.8 for p in PERMISSIONS},
    "risk_based_07_05_03":   {p: tier_thr(p, 0.7, 0.5, 0.3) for p in PERMISSIONS},
    "risk_based_06_04_02":   {p: tier_thr(p, 0.6, 0.4, 0.2) for p in PERMISSIONS},
    "risk_based_05_03_01":   {p: tier_thr(p, 0.5, 0.3, 0.1) for p in PERMISSIONS},
    "risk_based_08_06_04":   {p: tier_thr(p, 0.8, 0.6, 0.4) for p in PERMISSIONS},
}

W = {1: 3, 2: 2, 3: 1}
weights_per_perm = np.array([W[TOOL_TIERS[p]] for p in PERMISSIONS])
test = json.load(open("dataset/test.json"))
gt   = np.array([[r["permissions"].get(p, False) for p in PERMISSIONS] for r in test], dtype=int)

print(f"\n{'config':<24}{'sev-d':>7}{'over':>7}{'under':>7}{'macroP':>9}{'macroF1':>9}{'auto':>6}{'sev/auto':>10}")
print('-' * 79)

# also reference rows
def metrics(preds):
    n = preds.shape[0]
    over_mask  = (preds == 1) & (gt == 0)
    under_mask = (preds == 0) & (gt == 1)
    sev_d  = (over_mask * weights_per_perm).sum(axis=1).mean()
    o_rate = over_mask.any(axis=1).mean()
    u_rate = under_mask.any(axis=1).mean()
    auto   = ~under_mask.any(axis=1)
    sev_auto = (over_mask[auto] * weights_per_perm).sum(axis=1).mean() if auto.any() else 0
    tp = ((gt == 1) & (preds == 1)).sum(axis=0)
    fp = ((gt == 0) & (preds == 1)).sum(axis=0)
    fn = ((gt == 1) & (preds == 0)).sum(axis=0)
    P = np.array([tp[j] / max(tp[j]+fp[j], 1) for j in range(15)])
    R = np.array([tp[j] / max(tp[j]+fn[j], 1) for j in range(15)])
    F1 = np.array([2*P[j]*R[j]/max(P[j]+R[j], 1e-9) for j in range(15)])
    return sev_d, o_rate, u_rate, P.mean(), F1.mean(), int(auto.sum()), sev_auto

# C0, C1 references for the eye
print(f"{'C0 (all-grant ref)':<24}{'26.12':>7}{'100%':>7}{'0%':>7}{'—':>9}{'—':>9}{100:>6}{'26.12':>10}")
print(f"{'C1 (role ceiling)':<24}{'17.51':>7}{'100%':>7}{'2%':>7}{'—':>9}{'—':>9}{98:>6}{'~17.5':>10}")

for name, thr_map in CONFIGS.items():
    thr_vec = np.array([thr_map[p] for p in PERMISSIONS])
    preds = (probs >= thr_vec).astype(int)
    sev_d, o, u, mP, mF1, auto, sev_a = metrics(preds)
    star = " ★" if (u < 0.10 and mP >= 0.89) else ""
    print(f"{name:<24}{sev_d:>7.2f}{o:>7.1%}{u:>7.1%}{mP:>9.3f}{mF1:>9.3f}{auto:>6}{sev_a:>10.2f}{star}")

Saved probability matrix → roberta_test_probs.npy ((100, 15))

config                    sev-d   over  under   macroP  macroF1  auto  sev/auto
-------------------------------------------------------------------------------
C0 (all-grant ref)        26.12   100%     0%        —        —   100     26.12
C1 (role ceiling)         17.51   100%     2%        —        —    98     ~17.5
static_05                  0.89  33.0%  24.0%    0.853    0.848    76      0.79
static_08                  0.37  14.0%  42.0%    0.913    0.810    58      0.36
risk_based_07_05_03        0.72  35.0%  26.0%    0.858    0.848    74      0.74
risk_based_06_04_02        1.12  49.0%  21.0%    0.793    0.825    79      1.15
risk_based_05_03_01        1.61  66.0%  16.0%    0.740    0.803    84      1.67
risk_based_08_06_04        0.60  29.0%  30.0%    0.873    0.844    70      0.64


## Exclude email_send_external

In [43]:
import json, numpy as np
from mostargate.constants import TOOLS, TOOL_TIERS

PERMISSIONS = list(TOOLS.keys())
EXCLUDE = "email_send_external"
keep_idx = [j for j, p in enumerate(PERMISSIONS) if p != EXCLUDE]
keep_perms = [PERMISSIONS[j] for j in keep_idx]

# Slice probs and ground truth to drop the excluded permission
probs_keep = probs[:, keep_idx]
test = json.load(open("dataset/test.json"))
gt_full = np.array([[r["permissions"].get(p, False) for p in PERMISSIONS] for r in test], dtype=int)
gt_keep = gt_full[:, keep_idx]
weights = np.array([{1:3,2:2,3:1}[TOOL_TIERS[p]] for p in keep_perms])

def tier_thr(p, t1, t2, t3):
    return {1: t1, 2: t2, 3: t3}[TOOL_TIERS[p]]
CONFIGS = {
    "static_05":             {p: 0.5 for p in keep_perms},
    "static_08":             {p: 0.8 for p in keep_perms},
    "risk_based_07_05_03":   {p: tier_thr(p, 0.7, 0.5, 0.3) for p in keep_perms},
    "risk_based_06_04_02":   {p: tier_thr(p, 0.6, 0.4, 0.2) for p in keep_perms},
    "risk_based_05_03_01":   {p: tier_thr(p, 0.5, 0.3, 0.1) for p in keep_perms},
    "risk_based_08_06_04":   {p: tier_thr(p, 0.8, 0.6, 0.4) for p in keep_perms},
}

def metrics(preds):
    over_mask  = (preds == 1) & (gt_keep == 0)
    under_mask = (preds == 0) & (gt_keep == 1)
    sev_d  = (over_mask * weights).sum(axis=1).mean()
    o_rate = over_mask.any(axis=1).mean()
    u_rate = under_mask.any(axis=1).mean()
    auto   = ~under_mask.any(axis=1)
    sev_auto = (over_mask[auto] * weights).sum(axis=1).mean() if auto.any() else 0
    tp = ((gt_keep == 1) & (preds == 1)).sum(axis=0)
    fp = ((gt_keep == 0) & (preds == 1)).sum(axis=0)
    fn = ((gt_keep == 1) & (preds == 0)).sum(axis=0)
    N = len(keep_perms)
    P = np.array([tp[j] / max(tp[j]+fp[j], 1) for j in range(N)])
    R = np.array([tp[j] / max(tp[j]+fn[j], 1) for j in range(N)])
    F1 = np.array([2*P[j]*R[j]/max(P[j]+R[j], 1e-9) for j in range(N)])
    return sev_d, o_rate, u_rate, P.mean(), F1.mean(), int(auto.sum()), sev_auto

print(f"Excluding {EXCLUDE!r}: {len(keep_perms)} permissions remain")
print(f"\n{'config':<24}{'sev-d':>7}{'over':>7}{'under':>7}{'macroP':>9}{'macroF1':>9}{'auto':>6}{'sev/auto':>10}")
print('-' * 79)
for name, thr_map in CONFIGS.items():
    thr_vec = np.array([thr_map[p] for p in keep_perms])
    preds = (probs_keep >= thr_vec).astype(int)
    sev_d, o, u, mP, mF1, auto, sev_a = metrics(preds)
    star = " ★" if (u < 0.10 and mP >= 0.89) else ""
    print(f"{name:<24}{sev_d:>7.2f}{o:>7.1%}{u:>7.1%}{mP:>9.3f}{mF1:>9.3f}{auto:>6}{sev_a:>10.2f}{star}")

Excluding 'email_send_external': 14 permissions remain

config                    sev-d   over  under   macroP  macroF1  auto  sev/auto
-------------------------------------------------------------------------------
static_05                  0.77  32.0%  21.0%    0.874    0.867    79      0.68
static_08                  0.31  12.0%  38.0%    0.931    0.827    62      0.24
risk_based_07_05_03        0.66  34.0%  22.0%    0.872    0.867    78      0.63
risk_based_06_04_02        1.03  48.0%  17.0%    0.809    0.846    83      1.02
risk_based_05_03_01        1.49  65.0%  13.0%    0.753    0.818    87      1.51
risk_based_08_06_04        0.54  27.0%  26.0%    0.888    0.863    74      0.53
